In [ ]:
import numpy as np
import pandas as pd
from astropy import cosmology as apc
from phytorch.cosmology.core import H100
from phytorch.cosmology.drivers.analytic import FlatLambdaCDM
from scipy.interpolate import interp1d

from libech0ez import fitres_to_data, ztf_offz, ztf_off
from libech0ez.vpec import correct_cmb, correct_vpec, vpec_cov

fidcosmo = FlatLambdaCDM(Om0=0.3, H0=0.7*H100)

In [ ]:
FIT_NAME = 'nom'
CALIB_NAME = f'pplus-cdd_{FIT_NAME}'
ZTF_NAME = f'ztfdr2-vlhq_{FIT_NAME}'
ZTF_PC = True
ZTF_SM = True

In [ ]:
from typing import Literal
from itertools import product


def get_ztf(ztf_name, pc: Literal['pcorr', 'pcov', 'pborg'], sm):
    ztf = pd.read_csv(f'data/{ztf_name}.csv').set_index('ztfname').rename(columns={'z': 'zHEL'})

    ztfgz = pd.read_csv('data/ztfdr2/ztfdr2+gz-desi.csv').sort_values('angDist').drop_duplicates('ztfname').set_index('ztfname')[['SFFDF', 'DEONo', 'SAYes']]
    ztf = ztf.assign(is_late=(ztfgz['SFFDF'] >= 0.27) * (ztfgz['DEONo'] > 0.68))
    ztf['is_late'] = ztf['is_late'] == True

    for name in 'm', 'x1':
        pocket = pd.read_csv(f'data/ztfdr2/pocket/pocket-{name}.csv')
        ztf[f'{name}_pocketcorr'] = np.where(
            ztf['t0'] < 58788,  # 1 Nov 2019
            ztf[name],
            interp1d(pocket[f'true'] + pocket[f'delta'], pocket['true'], fill_value='extrapolate')(ztf[name])
        )

    ztf['m_smp'] = ztf['m_pocketcorr'] + interp1d(ztf_offz, ztf_off)(ztf['zHEL'])

    ztf = correct_cmb(ztf)

    if pc == 'pcorr':
        ztf, _ = correct_vpec(ztf, 'data/twompp_velocity.npy')
        z_cov = np.eye(len(ztf)) * (ztf['zHEL_err'].to_numpy()**2 + (300/3e5)**2)
    elif pc == 'pcov':
        Cvv = vpec_cov(
            np.deg2rad(ztf['ra_host']).to_numpy(), np.deg2rad(ztf['dec_host']).to_numpy(),
            apc.FlatLambdaCDM(Om0=0.3, H0=70).comoving_distance(ztf['zHD']).to('Mpc').value
        )
        z_cov = np.outer(*2*(1+ztf['zHD'].to_numpy(),)) * (
            Cvv/3e5**2 / np.outer(*2*(1+ztf['zPEC'].to_numpy(),))
            + np.eye(len(ztf)) * (ztf['zHEL_err']**2 / (1+ztf['zHEL'])).to_numpy()
        )
    elif pc == 'pborg':
        manticore = np.load(f'data/{ztf_name}-manticore.npz')
        ztf['zHD'], z_cov = map(np.array, (manticore['zHD'], manticore['z_cov']))
    else:
        raise ValueError(pc)

    data_ztf = fitres_to_data(ztf, True, {
        'z': 'zCMB' if pc == 'pcov' else 'zHD',
        'z_err': 'zHEL_err',
        'm': 'm_smp' if sm else 'm',
        'x1': 'x1_pocketcorr' if sm else 'x1',
        'logmass': 'mass', 'logmass_err': 'mass_err'
    }, fidcosmo, ('is_late',))

    mask_finite = np.isfinite(data_ztf['z'])
    return {key: val[mask_finite] for key, val in data_ztf.items()} | {
        'z_cov': z_cov[np.ix_(mask_finite, mask_finite)]
    }


def get_calib(calib_name):
    calib = pd.read_csv(f'data/{calib_name}.csv')

    data_calib = fitres_to_data(calib, False, {'z_err': 'zHDERR'})
    data_calib['distmod'] = calib['HOST_DIST'].to_numpy()
    # data_calib['distmod_err'] = calib['HOST_DISTERR'].to_numpy()

    calib_hosts = dict(np.load('data/pplus-cdd-hosts.npz', allow_pickle=True))
    hostlist = calib_hosts.pop('hostlist')
    calib_hosts['calib_idx'] = pd.Series(np.arange(len(hostlist), dtype=int), hostlist).loc[calib['HOST']].to_numpy()

    return data_calib, calib_hosts


def produce_pkgs(fit_name, ztf_pc, ztf_sm):
    ztf_name = f'ztfdr2-vlhq_{fit_name}'
    calib_name = f'pplus-cdd_{fit_name}'

    data_ztf = get_ztf(ztf_name, ztf_pc, ztf_sm)
    data_calib, calib_hosts = get_calib(calib_name)

    pkg = {
        key: np.concat([_data[key] for _data in (data_calib, data_ztf)]) for key in (
            'obs', 'cov', 'hf_mask',
            'logmass', 'logmass_err'
        )
    } | {'z': data_ztf['z'], 'z_cov': data_ztf['z_cov'], 'distmod_err': data_ztf['distmod_err']}

    mask_late = data_ztf['is_late']
    mask_highz = data_ztf['z'] > 0.025
    mask_lowz = data_ztf['z'] < 0.05

    pkg_late, pkg_highz, pkg_latehz, pkg_lowz = (
        {key: (
            val[np.ix_(mask, mask)] if val.shape == 2*(len(mask),)
            else val[mask] if len(val) == len(mask)
            else val[np.concat((np.ones(len(data_calib['z']), dtype=bool), mask))]
        ) for key, val in pkg.items()}
        for mask in (mask_late, mask_highz, mask_late * mask_highz, mask_lowz)
    )

    name = f'{calib_name}+{ztf_name}-{ztf_pc}{"-sm" if ztf_sm else ""}'
    np.savez(f'data/pkg/{name}.npz', **pkg, **calib_hosts, is_late=mask_late, is_highz=mask_highz)
    np.savez(f'data/pkg/{name}+late.npz', **pkg_late, **calib_hosts)
    np.savez(f'data/pkg/{name}+highz.npz', **pkg_highz, **calib_hosts)
    np.savez(f'data/pkg/{name}+late+highz.npz', **pkg_latehz, **calib_hosts)
    np.savez(f'data/pkg/{name}+lowz.npz', **pkg_lowz, **calib_hosts)


for fit_name in ('nom', *map('-'.join, product(('B22', 'T21'), ('CCM89', 'F99')))):
    produce_pkgs(fit_name, 'pcorr', True)
produce_pkgs('nom', 'pcov', True)
produce_pkgs('nom', 'pborg', True)
produce_pkgs('nom', 'pcorr', False)